In [39]:
import re
import os
import jpype
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.model_selection import learning_curve
from sklearn.neighbors import KNeighborsClassifier

ZEMBEREK_PATH="zemberek-full.jar"

In [40]:
os.environ["JAVA_HOME"]=r"C:\Program Files\Java\jdk-17"
if not jpype.isJVMStarted():
    jpype.startJVM(
        jpype.getDefaultJVMPath(),
        "-ea",
        f"-Djava.class.path={ZEMBEREK_PATH}",
        ignoreUnrecognized=True
    )

In [41]:
nltk.download('stopwords')
TurkishSentenceExtractor=jpype.JClass('zemberek.tokenization.TurkishSentenceExtractor')
TurkishMorphology=jpype.JClass('zemberek.morphology.TurkishMorphology')
TurkishSpellChecker=jpype.JClass('zemberek.normalization.TurkishSpellChecker')
TurkishTokenizer=jpype.JClass('zemberek.tokenization.TurkishTokenizer')
TurkishLexer=jpype.JClass('zemberek.tokenization.antlr.TurkishLexer')

extractor=TurkishSentenceExtractor.DEFAULT
morphology=TurkishMorphology.createWithDefaults()
tokenizer=TurkishTokenizer.ALL
spell=TurkishSpellChecker(morphology)
stop_words=nltk.corpus.stopwords.words('turkish')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\zynpe\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [42]:
def preprocess(text):
    if isinstance(text,list):
        if len(text)>0:
            text=text[0] 
        else:
            return ""
    text=re.sub(r'\b\d+\b','',text)
    text=re.sub(r'[^\w\s]','',text)
    text=re.sub(r'\s+',' ',text).strip()
    tokens=tokenizer.tokenize(text)
    process_word=[]

    for token in tokens:
        word=str(token.getText())
        if word.isalpha() and word.lower() not in stop_words:
            analysis=morphology.analyze(word)
            analysis_list=list(analysis)
            if len(analysis_list)>0:
                best_word=analysis_list[0]
                lemma=str(best_word.getStems().get(0))
                process_word.append(lemma)
            else:
                process_word.append(word)
    return " ".join(process_word)

In [43]:
df=pd.read_csv('data/dataset.csv')
print(f'Toplam Makale Sayısı: {len(df)}')
display(df['label'].value_counts())
display(df.head(2))

Toplam Makale Sayısı: 6430


label
tarih                   1498
saglik                  1497
bilgisayar_bilimleri    1495
finans                  1495
spor                     445
Name: count, dtype: int64

,id,label,journal,year,title,abstract,keywords,url
0,oai:dergipark.org.tr:article/1001762,bilgisayar_bilimleri,adyujsci,2021.0,Yansımalı Banach Uzaylarda Operatörlerin Çarpı...,Bu çalışmada iki problem incelenmiştir. Genel ...,Mathematical Sciences; Matematik; Dual s- numb...,https://dergipark.org.tr/en/pub/adyujsci/artic...
1,oai:dergipark.org.tr:article/1003333,bilgisayar_bilimleri,adyujsci,2021.0,CMS Açık Veri ile Eğitici Higgs Çalışması,"Kompakt Müon Solenoid (CMS) Deneyi, CERN’in Bü...",Nuclear Physics; Nükleer Fizik; : CMS Açık Ver...,https://dergipark.org.tr/en/pub/adyujsci/artic...


In [44]:
df['full_text']=df['title'].fillna('')+". "+df['abstract'].fillna('')
df['processed_text']=df['full_text'].apply(preprocess)
display(df[['full_text','processed_text']].tail(2))

,full_text,processed_text
6428,ATATÜRK ULUSLARARASI BARIŞ ÖDÜLÜ. Savaşların n...,atatürk uluslararası barış ödül savaş ol yıkı ...
6429,TARİHSEL OLAYLARI ÇÖZÜMLEMEDE METİN MADENCİLİĞ...,tarih olay çözümle metin maden nutuk üzerine b...


In [45]:
x_raw=df['processed_text'].values
y=df['label'].values
x_processed=[preprocess(text) for text in x_raw]
x_train,x_test,y_train,y_test=train_test_split(
    x_processed,y,test_size=0.2,random_state=42,stratify=y
)
print(f"Eğitim kümesi boyutu: {len(x_train)}")
print(f"Test kümesi boyutu: {len(x_test)}")
vectorizer=CountVectorizer(max_features=5000,ngram_range=(1,2))
x_train_vectorized=vectorizer.fit_transform(x_train)
x_test_vecrorized=vectorizer.transform(x_test)
print(f"Vektör boyutları:",x_train_vectorized.shape)

Eğitim kümesi boyutu: 5144
Test kümesi boyutu: 1286
Vektör boyutları: (5144, 5000)


In [49]:
knn_model=KNeighborsClassifier(n_neighbors=5,metric='cosine',n_jobs=1)
knn_model.fit(x_train_vectorized,y_train)
y_pred=knn_model.predict(x_test_vecrorized)
accuracy=accuracy_score(y_test,y_pred)
print(f"KNN model doğruluk oranı: ½{accuracy*100:.2f}\n")
print(classification_report(y_test,y_pred))

KNN model doğruluk oranı: ½85.85

                      precision    recall  f1-score   support

bilgisayar_bilimleri       0.85      0.92      0.88       299
              finans       0.87      0.80      0.83       299
              saglik       0.91      0.87      0.89       299
                spor       0.89      0.66      0.76        89
               tarih       0.81      0.90      0.85       300

            accuracy                           0.86      1286
           macro avg       0.87      0.83      0.84      1286
        weighted avg       0.86      0.86      0.86      1286



In [52]:
# Test etmek istediğin o özel, karmaşık ve niş metni buraya yaz:
ornek_metin = "Bu metin örneğin bilgisayar bilimleri ve sağlık alanını birleştiren biyoinformatik veya yapay zeka tabanlı bir tıbbi teşhis makalesi özeti olabilir..."

# 1. Metni ön işlemden geçir (Zemberek kök bulma ve temizlik adımları)
islenmis_metin = preprocess(ornek_metin)

# 2. Vektörleştir (Eğitim aşamasında kullanılan vectorizer üzerinden transform ediyoruz)
ornek_vec = vectorizer.transform([islenmis_metin])

# 3. KNN Modeli ile Tahmin Yap
tahmin_etiket = knn_model.predict(ornek_vec)[0]

# 4. Sınıf Olasılıklarını Görme (Hangi sınıfa ne kadar yakın olduğunu görmek için)
tahmin_olasilik = knn_model.predict_proba(ornek_vec)

print(f"--- KNN MODELİ TEST SONUCU ---")
print(f"Orijinal Metin: {ornek_metin}")
print(f"İşlenmiş Hali: {islenmis_metin}")
print(f"Tahmin Edilen Kategori: **{tahmin_etiket}**")

# Modelin sınıflara göre olasılık/uzaklık skorlarını da yazdıralım
print("\nSınıf İhtimal Dağılımı:")
for sinif, olasilik in zip(knn_model.classes_, tahmin_olasilik[0]):
    print(f"- {sinif}: %{olasilik * 100:.2f}")

--- KNN MODELİ TEST SONUCU ---
Orijinal Metin: Bu metin örneğin bilgisayar bilimleri ve sağlık alanını birleştiren biyoinformatik veya yapay zeka tabanlı bir tıbbi teşhis makalesi özeti olabilir...
İşlenmiş Hali: metin örneğin bilgisayar bilim sağlık alan birleş biyoinformatik yapay zeka taban bir tıbbi teşhis makale özet ol
Tahmin Edilen Kategori: **finans**

Sınıf İhtimal Dağılımı:
- bilgisayar_bilimleri: %0.00
- finans: %60.00
- saglik: %20.00
- spor: %0.00
- tarih: %20.00


In [55]:
dosyaadi="data/dataset.csv"
data=pd.read_csv(dosyaadi)
data.columns.tolist()



['id', 'label', 'journal', 'year', 'title', 'abstract', 'keywords', 'url']

In [56]:
drop_columns=['id','year','keywords','url']
data.drop(columns=drop_columns, inplace=True)
data.to_csv(dosyaadi,index=False,encoding='utf-8')
print("kalanlar", data.columns.tolist())

C:\Users\zynpe\AppData\Local\Temp\ipykernel_22988\2526017190.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  data.drop(columns=drop_columns, inplace=True)


kalanlar ['label', 'journal', 'title', 'abstract']


In [57]:
data.rename(columns={'label':'category'},inplace=True)
data.to_csv(dosyaadi,index=False,encoding='utf-8')
print("kalanlar", data.columns.tolist())

C:\Users\zynpe\AppData\Local\Temp\ipykernel_22988\963156079.py:1: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  data.rename(columns={'label':'category'},inplace=True)


kalanlar ['category', 'journal', 'title', 'abstract']
